# Build all six ablation indices on a free Kaggle T4

Locally these six embedding passes cost **~7.3 hours** of thermally-throttled
CPU (see `docs/HARDWARE.md`). The models are 22M-109M-parameter encoders, which
a T4 is idle-fast on: projected **~5 minutes** for all six.

## The rule this notebook is built around

> **Quality metrics (recall, MRR, nDCG) are hardware-independent** - same
> weights, same math, same answer. Compute them wherever is fastest.
> **Latency metrics are hardware-specific.** Measure them on one stated
> platform and never mix platforms within a table.

Reported explicitly ("quality on T4, latency on a documented CPU baseline")
this is more rigorous than hiding it. Reported carelessly it is dishonest. So
this notebook records `embed_platform` in every `config.json` it writes, and
`scripts/import_embeddings.py` keeps that provenance attached.

## What makes the output trustworthy

A `.npy` matrix carries no evidence about which chunks it describes. Row *i*
belongs to chunk *i* and nothing else records that pairing - so if this notebook
chunked even slightly differently from the local build, the vectors would still
load, still search, and still produce plausible numbers for the wrong passages.

Two things prevent that:

1. **The chunking code is imported, not reimplemented.** The dataset carries the
   repo's own `emailrag` package, so `chunk_corpus` here is byte-identical to
   `chunk_corpus` locally. Retyping the chunker into a notebook cell is the
   single most likely way to produce a subtly different corpus.
2. **`import_embeddings.py` re-chunks locally and requires exact, ordered chunk-id
   equality** before it will accept anything.

## Setup (one-time)

1. Phone-verify the Kaggle account - notebook internet access needs it, and
   pulling HF model weights needs internet.
2. Create a **private** Kaggle Dataset containing:
   - `sample.parquet` (from `data/interim/`, 53 MB)
   - the `emailrag/` package directory (from `src/`)

   Enron is public data, so a private dataset of it is fine. **Never upload
   Gmail-derived data** - see phase 7 in `TODO.md`.
3. Attach it to this notebook, set accelerator to **GPU T4 x2**, internet **on**.
4. Run all. Then `Save Version` -> the `/kaggle/working` output becomes a
   downloadable dataset.
5. Locally:
   ```bash
   kaggle kernels output <user>/<notebook-slug> -p ~/Downloads/kaggle-emailrag
   python scripts/import_embeddings.py --from ~/Downloads/kaggle-emailrag
   ```

In [ ]:
# Kaggle ships a recent CUDA torch. The torch==2.2.2 pin in requirements.txt is
# a macOS-x86 constraint only (PyTorch stopped publishing macosx_x86_64 wheels
# after 2.2.2) and must NOT be applied here - it would install a CPU build and
# throw away the entire point of the session.
!pip install -q sentence-transformers 2>&1 | tail -2

import torch, platform, subprocess
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then re-run.")

In [ ]:
import sys, json, time, os
from pathlib import Path

# Point these at wherever the attached dataset landed. Kaggle mounts datasets
# read-only under /kaggle/input/<dataset-slug>/.
INPUT = Path("/kaggle/input")
candidates = list(INPUT.rglob("sample.parquet"))
assert candidates, f"sample.parquet not found under {INPUT} - is the dataset attached?"
SAMPLE = candidates[0]

pkg = list(INPUT.rglob("emailrag/chunking/strategies.py"))
assert pkg, ("the emailrag package is not in the dataset. Upload src/emailrag/ "
             "alongside sample.parquet - the chunker must be the repo's own code, "
             "not a copy retyped into this notebook.")
SRC = pkg[0].parents[2]
sys.path.insert(0, str(SRC))

OUT = Path("/kaggle/working")
print("sample :", SAMPLE, f"({SAMPLE.stat().st_size/1e6:.0f} MB)")
print("package:", SRC)

In [ ]:
from emailrag.chunking import strategies as S
from emailrag.index import chunktext as CT

# Exactly the matrix the Makefile defines: one factor at a time, not a full grid.
#   dimension 1 - four chunkings against the cheap baseline model
#   dimension 2 - two more models against the chunking that wins dimension 1
# Until dimension 1 has a winner, embed dimension 2 against `thread_aware`; on a
# T4 the whole set is minutes, so building the extra configs is cheaper than a
# second session.
BASELINE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNKERS = ["fixed_512", "fixed_512_ov64", "whole_message", "thread_aware"]
EXTRA_MODELS = ["BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5"]
BEST_CHUNKING = "thread_aware"

CONFIGS = [(c, BASELINE_MODEL) for c in CHUNKERS] + \
          [(BEST_CHUNKING, m) for m in EXTRA_MODELS]
for c, m in CONFIGS:
    print(f"  {c:16s} {m}")

In [ ]:
# Chunk once per strategy and reuse across models. The reference tokenizer is
# fixed for all strategies and all models (see chunking/strategies.py): chunking
# with each model's own tokenizer would make dimensions 1 and 2 interact.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(S.REFERENCE_TOKENIZER)
chunked = {}
for chunking in sorted({c for c, _ in CONFIGS}):
    t0 = time.time()
    chunks = CT.rebuild_chunks(SAMPLE, chunking, tokenizer)
    chunked[chunking] = chunks
    n_tok = [c.n_tokens for c in chunks]
    print(f"{chunking:16s} {len(chunks):>7,} chunks  "
          f"mean {sum(n_tok)/len(n_tok):5.1f} tok  max {max(n_tok):4d}  "
          f"{time.time()-t0:5.0f}s")

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

SHARD_SIZE = 20_000          # matches index/embed.py, so shards are interchangeable
BATCH_SIZE = 256             # a T4 has headroom the 32 used on CPU does not need


def build(chunking: str, model_id: str, model) -> dict:
    chunks = chunked[chunking]
    texts = [c.text for c in chunks]
    ids = [c.chunk_id for c in chunks]
    dest = OUT / f"{chunking}__{model_id.split('/')[-1]}"
    (dest / "dense").mkdir(parents=True, exist_ok=True)
    (dest / "shards").mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    shards = []
    for offset in range(0, len(texts), SHARD_SIZE):
        block = texts[offset:offset + SHARD_SIZE]
        # normalize_embeddings=True is not optional: DenseIndex treats the dot
        # product AS the cosine, and import_embeddings.py rejects a matrix whose
        # rows are not unit length.
        vecs = model.encode(block, batch_size=BATCH_SIZE, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=False
                            ).astype(np.float32)
        np.save(dest / "shards" / f"shard_{offset:07d}.npy", vecs)
        shards.append(vecs)
        print(f"    {min(offset+SHARD_SIZE, len(texts)):>7,}/{len(texts):,}", end="\r")
    secs = time.time() - t0

    matrix = np.vstack(shards)
    np.save(dest / "dense" / "embeddings.npy", matrix)
    (dest / "dense" / "chunk_ids.txt").write_text("\n".join(ids))

    with open(dest / "chunks.jsonl", "w") as fh:
        for c in chunks:
            fh.write(json.dumps({"chunk_id": c.chunk_id, "dedup_key": c.dedup_key,
                                 "thread_id": c.thread_id,
                                 "n_tokens": c.n_tokens}) + "\n")

    n_tok = [c.n_tokens for c in chunks]
    meta = {
        "chunking": chunking,
        "model": model_id,
        "dim": int(matrix.shape[1]),
        "n_messages": N_MESSAGES,
        "n_chunks": len(chunks),
        "chunks_per_message": round(len(chunks) / N_MESSAGES, 3),
        "mean_tokens": round(sum(n_tok) / len(n_tok), 1),
        "embed_seconds": round(secs, 1),
        "embed_docs_per_sec": round(len(texts) / secs, 2),
        # Provenance, so a GPU throughput number can never end up in a table of
        # CPU numbers.
        "embed_platform": f"kaggle-{torch.cuda.get_device_name(0).replace(' ', '-')}",
        "torch": torch.__version__,
        "batch_size": BATCH_SIZE,
    }
    (dest / "config.json").write_text(json.dumps(meta, indent=2))
    print(f"    {len(texts):,} chunks in {secs/60:5.2f} min "
          f"({len(texts)/secs:7.1f} docs/s)  -> {dest.name}")
    return meta


N_MESSAGES = len({c.dedup_key for c in chunked[BEST_CHUNKING]})
print("messages:", f"{N_MESSAGES:,}")

In [ ]:
# Grouped by model so each set of weights is loaded once.
results = []
by_model = {}
for chunking, model_id in CONFIGS:
    by_model.setdefault(model_id, []).append(chunking)

for model_id, chunkings in by_model.items():
    print(f"\n=== {model_id} ===")
    model = SentenceTransformer(model_id, device="cuda")
    for chunking in chunkings:
        results.append(build(chunking, model_id, model))
    del model
    torch.cuda.empty_cache()

total = sum(r["embed_seconds"] for r in results)
print(f"\n{len(results)} indices in {total/60:.1f} min total")
print("local projection for the same work was ~7.3 h - see docs/HARDWARE.md")

In [ ]:
# Self-check before saving a version. These are the same invariants
# import_embeddings.py enforces locally; failing here saves a download.
import numpy as np

ok = True
for meta in results:
    d = OUT / f"{meta['chunking']}__{meta['model'].split('/')[-1]}"
    ids = (d / "dense" / "chunk_ids.txt").read_text().splitlines()
    m = np.load(d / "dense" / "embeddings.npy", mmap_mode="r")
    norms = np.linalg.norm(np.asarray(m[:1000]), axis=1)
    checks = {
        "rows == ids": m.shape[0] == len(ids),
        "ids unique": len(set(ids)) == len(ids),
        "dim matches config": m.shape[1] == meta["dim"],
        "unit norms": bool(np.allclose(norms, 1.0, atol=1e-3)),
        "float32": m.dtype == np.float32,
    }
    bad = [k for k, v in checks.items() if not v]
    ok &= not bad
    print(f"{d.name:44s} {'OK' if not bad else 'FAILED: ' + ', '.join(bad)}")
    print(f"{'':44s} {m.shape[0]:,} x {m.shape[1]}, "
          f"{m.nbytes/1e6:.0f} MB")

print("\nall checks passed" if ok else "\nFIX THE FAILURES ABOVE BEFORE SAVING")
!du -sh /kaggle/working

## Then, locally

```bash
kaggle kernels output <user>/<notebook-slug> -p ~/Downloads/kaggle-emailrag
python scripts/import_embeddings.py --from ~/Downloads/kaggle-emailrag
make bench --dimension chunking
```

`import_embeddings.py` re-chunks the local corpus and refuses to import unless
every chunk id matches, in order. It builds the BM25 side locally (CPU-cheap, no
GPU session needed) and records `embed_platform` so the GPU throughput figures
here never appear in a CPU latency table.

**A note on the `shards/` directories.** They are written for parity with the
local checkpointing format in `index/embed.py` and are redundant with
`dense/embeddings.npy`. If the output dataset is uncomfortably large, delete
`*/shards` before saving the version - the import path only reads `dense/`.